In [4]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

from collections import Counter
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader


# ============================================================
# LSTM CUSTOMER REVIEW SENTIMENT ANALYZER
# ============================================================

print("=" * 60)
print("LSTM CUSTOMER REVIEW SENTIMENT ANALYZER")
print("=" * 60)


# ------------------------------------------------------------
# 1. LOAD DATASET
# ------------------------------------------------------------

df = pd.read_csv("IMDB Dataset.csv")

print("\nDataset Loaded")
print("Total Reviews :", len(df))

print("\nFirst 5 Rows")
print(df.head())


# ------------------------------------------------------------
# 2. CONVERT SENTIMENT TO NUMBERS
# ------------------------------------------------------------

df["sentiment"] = df["sentiment"].map({
    "negative": 0,
    "positive": 1
})


reviews = df["review"].to_numpy()
labels = df["sentiment"].to_numpy()


# ------------------------------------------------------------
# 3. SPLIT DATASET
# ------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    reviews,
    labels,
    test_size=0.20,
    random_state=42,
    stratify=labels
)

print("\nDataset Split")
print("Training Reviews :", len(X_train))
print("Testing Reviews  :", len(X_test))


# ------------------------------------------------------------
# 4. TOKENIZATION
# ------------------------------------------------------------

def tokenize(text):
    text = text.lower()

    text = text.replace("<br />", " ")
    text = text.replace("<br>", " ")
    text = text.replace(".", " ")
    text = text.replace(",", " ")
    text = text.replace("!", " ")
    text = text.replace("?", " ")
    text = text.replace(":", " ")
    text = text.replace(";", " ")
    text = text.replace('"', " ")
    text = text.replace("'", " ")

    return text.split()


# ------------------------------------------------------------
# 5. CREATE VOCABULARY
# ------------------------------------------------------------

word_count = Counter()

for review in X_train:
    words = tokenize(review)
    word_count.update(words)


MAX_VOCAB_SIZE = 10000

word_to_index = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in word_count.most_common(MAX_VOCAB_SIZE - 2):
    word_to_index[word] = len(word_to_index)


print("\nVocabulary Created")
print("Vocabulary Size :", len(word_to_index))


# ------------------------------------------------------------
# 6. CONVERT REVIEWS INTO SEQUENCES
# ------------------------------------------------------------

MAX_LENGTH = 200


def encode_review(review):

    words = tokenize(review)

    sequence = []

    for word in words:
        if word in word_to_index:
            sequence.append(word_to_index[word])
        else:
            sequence.append(word_to_index["<UNK>"])

    # Keep the last 200 words
    sequence = sequence[-MAX_LENGTH:]

    # Left padding
    if len(sequence) < MAX_LENGTH:
        sequence = [0] * (MAX_LENGTH - len(sequence)) + sequence

    return sequence


X_train_encoded = np.array(
    [encode_review(review) for review in X_train],
    dtype=np.int64
)

X_test_encoded = np.array(
    [encode_review(review) for review in X_test],
    dtype=np.int64
)


print("\nSequences Created")
print("Training Shape :", X_train_encoded.shape)
print("Testing Shape  :", X_test_encoded.shape)


# ------------------------------------------------------------
# 7. CONVERT TO PYTORCH TENSORS
# ------------------------------------------------------------

X_train_tensor = torch.tensor(X_train_encoded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_encoded, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)


# ------------------------------------------------------------
# 8. CREATE DATA LOADERS
# ------------------------------------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)


BATCH_SIZE = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)


# ------------------------------------------------------------
# 9. CREATE LSTM MODEL
# ------------------------------------------------------------

class SentimentLSTM(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            128,
            padding_idx=0
        )

        self.lstm = nn.LSTM(
            input_size=128,
            hidden_size=128,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(
            128,
            1
        )


    def forward(self, x):

        x = self.embedding(x)

        output, (hidden, cell) = self.lstm(x)

        hidden = hidden[-1]

        hidden = self.dropout(hidden)

        output = self.fc(hidden)

        return output.squeeze(1)


model = SentimentLSTM(
    len(word_to_index)
)


print("\nLSTM Model Created")


# ------------------------------------------------------------
# 10. LOSS FUNCTION AND OPTIMIZER
# ------------------------------------------------------------

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ------------------------------------------------------------
# 11. TRAIN MODEL
# ------------------------------------------------------------

EPOCHS = 5

print("\nTraining Started")

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for reviews_batch, labels_batch in train_loader:

        optimizer.zero_grad()

        predictions = model(reviews_batch)

        loss = criterion(
            predictions,
            labels_batch
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()


    average_loss = total_loss / len(train_loader)

    print(
        "Epoch",
        epoch + 1,
        "/",
        EPOCHS,
        "- Loss:",
        round(average_loss, 4)
    )


# ------------------------------------------------------------
# 12. EVALUATE MODEL
# ------------------------------------------------------------

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for reviews_batch, labels_batch in test_loader:

        predictions = model(reviews_batch)

        predictions = torch.sigmoid(predictions)

        predicted_labels = (
            predictions >= 0.5
        ).float()

        correct += (
            predicted_labels == labels_batch
        ).sum().item()

        total += labels_batch.size(0)


accuracy = correct / total


print("\n" + "=" * 60)
print("EVALUATION RESULTS")
print("=" * 60)

print("\nTest Accuracy :", round(accuracy, 4))

print(
    "Accuracy Percentage :",
    round(accuracy * 100, 2),
    "%"
)


# ------------------------------------------------------------
# 13. USER INPUT SENTIMENT PREDICTION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("USER INPUT SENTIMENT PREDICTION")
print("=" * 60)


user_review = input("\nEnter a movie review: ")


user_sequence = encode_review(user_review)

user_tensor = torch.tensor(
    [user_sequence],
    dtype=torch.long
)


model.eval()

with torch.no_grad():

    prediction = model(user_tensor)

    probability = torch.sigmoid(prediction).item()


if probability >= 0.5:

    sentiment = "Positive"

    confidence = probability * 100

else:

    sentiment = "Negative"

    confidence = (1 - probability) * 100


print("\nSentiment Result")

print("Review    :", user_review)

print("Sentiment :", sentiment)

print(
    "Confidence:",
    round(confidence, 2),
    "%"
)


print("\n" + "=" * 60)
print("LSTM SENTIMENT ANALYSIS COMPLETED")
print("=" * 60)

LSTM CUSTOMER REVIEW SENTIMENT ANALYZER

Dataset Loaded
Total Reviews : 50000

First 5 Rows
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Dataset Split
Training Reviews : 40000
Testing Reviews  : 10000

Vocabulary Created
Vocabulary Size : 10000

Sequences Created
Training Shape : (40000, 200)
Testing Shape  : (10000, 200)

LSTM Model Created

Training Started
Epoch 1 / 5 - Loss: 0.5758
Epoch 2 / 5 - Loss: 0.4388
Epoch 3 / 5 - Loss: 0.3535
Epoch 4 / 5 - Loss: 0.2927
Epoch 5 / 5 - Loss: 0.2529

EVALUATION RESULTS

Test Accuracy : 0.8692
Accuracy Percentage : 86.92 %

USER INPUT SENTIMENT PREDICTION



Enter a movie review:  This movie was excellent and I really enjoyed watching it.



Sentiment Result
Review    : This movie was excellent and I really enjoyed watching it.
Sentiment : Positive
Confidence: 97.73 %

LSTM SENTIMENT ANALYSIS COMPLETED
